In [ ]:
# import required libraries
import requests
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait
import time
import pyperclip
import os
import re
import json

In [ ]:
# create helper function to use selenium for dynamically rendered pages
def fetch_rendered_html(url: str, wait_for_selector: str = None, timeout: int = 15) -> BeautifulSoup:
    """
    Fetches a URL using headless Chrome, waits for JS execution,
    and returns a BeautifulSoup instance of the fully rendered DOM.
    """
    # 1. Configure Chrome Options for Headless Scraping
    options = Options()
    options.add_argument("--headless=new")  # Modern headless mode
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    
    # Spoof realistic User-Agent to avoid bot blocks
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/122.0.0.0 Safari/537.36"
    )

    # 2. Initialize Driver
    driver = webdriver.Chrome(options=options)

    try:
        driver.get(url)

        # 3. Handle JavaScript Wait Conditions
        if wait_for_selector:
            # Explicit Wait: Pauses until target dynamic element appears in the DOM
            WebDriverWait(driver, timeout).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, wait_for_selector))
            )
        else:
            # Fallback: Brief sleep for general JS script execution
            time.sleep(3)

        # 4. Extract the fully rendered DOM
        rendered_html = driver.page_source
        return BeautifulSoup(rendered_html, "html.parser")

    finally:
        # Always close the browser instance to prevent memory leaks
        driver.quit()

In [ ]:
# read in the `exploration` html directly from a URL and prettify it using BeautifulSoup, then write it to a file
link = input("Enter the URL of the site you want to scrape: ")
org = "tacomaparks"
static_page = False # use to switch between rendering dynamic or static content

# get the HTML content of the page
if static_page:
    response = requests.get(link)
    soup = BeautifulSoup(response.content, 'html.parser')
else:
    soup = fetch_rendered_html(link)

In [ ]:
# filter to only the section of interest, which is the table containing the data we want to extract
tag_class = "tribe-common-l-container tribe-events-l-container"
# display(soup.find_all("section", class_=tag_class)[0])

In [ ]:
target_section = soup.find_all("section", class_=tag_class)[0]
target_table = target_section.find_all("div", "tribe-common-g-row tribe-events-calendar-list__event-row") # , class_="container container--md2") # IMPORTANT: this is the div that contains the table we want to extract

# strip out unnecessary scripts and styles
for script in target_section(["script", "style"]):
    script.decompose()

In [ ]:
# apply formatting to the HTML using BeautifulSoup's prettify method
pretty_html = target_section.prettify()

# write the prettified HTML to a file
if pretty_html:
    # write the prettified HTML to a file in the current working directory
    with open(f"{org}.html", "w", encoding="utf-8") as f:
        f.write(pretty_html)

In [ ]:
display(target_table[0])